# Complexity-06 — Simuler un circuit quantique classiquement

**Navigation** : [<< Complexity-05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) | [Index de la série](README.md) | [Version C#](Complexity-06-Simuler-Circuit-Quantique-Classiquement-CSharp.ipynb) | [Complexity-06b — Aaronson-Gottesman, stabilisation et déquantification >>](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb)

> **Public : Licence.** Le pont avec la déquantification et la stabilisation de Gottesman est
> approfondi dans le carnet [06b](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb) ;
> celui-ci compare deux représentations d’une distribution uniforme connue. Il ne mesure pas une borne inférieure générale de simulation.

## Ce que ce notebook suppose

- [05 — Compter est plus dur que vérifier : #P et la permanente](Complexity-05-CountingHarder-Permanent-Python.ipynb) :
  la classe **#P**, la distinction entre **décider** et **compter**, et la formule de Ryser ;
- [05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) :
  le **boson sampling**, dont les probabilités sont liées à des permanentes : échantillonner ne constitue pas un oracle de comptage exact ;
- manipuler une matrice de Pauli et un produit tensoriel.

## Ce que vous saurez faire à la fin

1. énoncer ce que signifie **simuler classiquement** un circuit quantique (sortie
   probabiliste sur n qubits = 2^n amplitudes complexes) ;
2. donner un **invariant de Pauli** sous tension : un circuit Clifford (H, S, CNOT) sur n
   qubits se décrit par un tableau de taille **O(n²)** et non 2^n amplitudes — algorithme de Gottesman-Knill ;
3. **mesurer** le coût d’un vecteur explicite et d’un tirage direct sur la même loi uniforme, sans les confondre avec une séparation de classes ; rappeler le statut
   **BQP ⊆ PP ⊆ P#P** (theoreme ; l’ouverte est BQP vs P / PH) ;
4. distinguer deux régimes de coût pour la simulation classique : **échantillonner** vs
   **calculer la distribution de probabilité exacte** (Aaronson-Arkhipov 2011 pour le boson sampling ; folklore pour les circuits universels) ;
5. énoncer ce que la **simulation classique** d'un circuit quantique universel nous
   apprend sur la frontière BQP/P.

**Durée estimée** : 45 minutes.

## 1. Ce que simuler classiquement signifie

Un circuit quantique à k portes et n qubits prépare un état

> |ψ⟩ = U_k · U_{k-1} · ... · U_1 · |0^n⟩

où chaque U_i est une porte (Hadamard, phase, CNOT, Toffoli…). L'état est un vecteur de
2^n amplitudes complexes ; la mesure du registre de n qubits renvoie une loi categorielle sur les 2^n chaînes de bits possibles
(la mesure d'un qubit seul projete sur la base computationnelle donne bien une Bernoulli à 2 issues,
mais c'est la mesure du registre qui est sur 2^n issues). **Simuler classiquement** le circuit, c'est
produire des échantillons de cette distribution — ou, plus exigeant, calculer exactement
les 2^n probabilités.

La question de la **simulation polynomiale** d'un circuit quantique arbitraire est l'une
des grandes questions ouvertes de la théorie : un algorithme déterministe polynomial décidant tous les langages BQP donnerait P = BQP. Un simulateur classique probabiliste polynomial reproduisant fidèlement leurs sorties, avec une précision préservant une erreur bornée, donnerait BPP = BQP ; il ne fournit pas à lui seul une décision sans hasard. Ici BPP désigne le temps polynomial probabiliste à erreur bornée, et P ⊆ BPP ⊆ BQP. Le coût d’un simulateur particulier ne démontre aucune séparation.

Ce notebook explore une **fenêtre** dans cette question : la famille **Clifford** (Hadamard,
phase, CNOT) — où la simulation classique redevient polynomiale par l'algorithme de
Gottesman-Knill (1998). En dehors de Clifford, on entre dans la zone où la simulation
naïve coûte O(2^n).

In [1]:
import time
import numpy as np
from math import factorial

print("numpy", np.__version__)
print("python", __import__("sys").version_info[:3])

numpy 2.4.6
python (3, 13, 15)


## 2. La barrière exponentielle : simuler naïvement

Un circuit à n qubits a 2^n amplitudes. Pour un circuit universel arbitraire (Hadamard +
T + CNOT par exemple), la méthode qui stocke toutes les amplitudes demande O(2^n) mémoire et un travail exponentiel pour les parcourir. Ce coût décrit cette représentation explicite, sans être une borne inférieure générale pour échantillonner. Pour la **mesure** sur un registre de n qubits, l'algorithme le plus naïf accumule
toutes les amplitudes au carré ; cette représentation devient coûteuse vers n ≈ 30, y compris pour une distribution dont une autre représentation serait simple.

Mesurons cette croissance : combien de temps demande une simulation vectorielle pour un
circuit de Hadamard uniquement (préparation d'une superposition uniforme) à n = 4, 8, 16,
20 qubits ?

In [2]:
def simuler_hadamard_naive(n_qubits, n_echantillons=1024):
    """Prepare |+>^n (Hadamard sur chaque qubit), mesure uniformement sur n qubits.

    Chrono englobant : construction du vecteur d'etat (coût O(2^n) en memoire)
    + calcul des probabilites + echantillonnage. C'est le coût global de la
    simulation naîve qu'on veut mesurer, pas seulement le tirage final.
    """
    dim = 1 << n_qubits
    rng = np.random.default_rng(seed=20261007)
    # Warm-up isole : exclut le coût de chargement de numpy / cache CPU
    _ = rng.choice(dim, size=64, p=np.ones(dim) / dim)
    t0 = time.perf_counter()
    # Construction du vecteur d'etat : O(2^n) memoire
    etat = np.ones(dim, dtype=np.complex128) / np.sqrt(dim)
    # Calcul des probabilites (autre O(2^n) memoire pour probas float64)
    probas = np.abs(etat) ** 2
    # Echantillonnage selon la distribution
    indices = rng.choice(dim, size=n_echantillons, p=probas)
    t1 = time.perf_counter()
    return t1 - t0

for n in (4, 8, 16, 20):
    t = simuler_hadamard_naive(n, n_echantillons=4096)
    print(f"n = {n:>2} qubits : {t*1000:7.2f} ms pour 4096 echantillons (chrono complet : construction etat + echantillonnage, dim 2^n = {1<<n})")

n =  4 qubits :    0.88 ms pour 4096 echantillons (chrono complet : construction etat + echantillonnage, dim 2^n = 16)
n =  8 qubits :    0.35 ms pour 4096 echantillons (chrono complet : construction etat + echantillonnage, dim 2^n = 256)
n = 16 qubits :    2.64 ms pour 4096 echantillons (chrono complet : construction etat + echantillonnage, dim 2^n = 65536)
n = 20 qubits :   38.41 ms pour 4096 echantillons (chrono complet : construction etat + echantillonnage, dim 2^n = 1048576)


Le coût mémoire du vecteur d'état croît comme 2^n nombres complexes (16 octets par
amplitude sur float64). Au-delà de n = 30 environ, la mémoire d'un poste de travail
standard (16 Go) déborde (2^30 × 16 o = 16 GiB, soit environ 17,2 Go décimaux, pour le vecteur d'état
seul ; les probas float64 et la marge système portent la limite pratique
vers n ≈ 28) ; au-delà de n = 40, il faut une machine HPC.

Cette barrière de représentation motive la recherche de descriptions plus compactes :
pour n suffisamment grand, le vecteur explicite dépasse les ressources disponibles. Cela ne démontre pas un avantage quantique : la loi uniforme du banc dispose précisément d’un tirage classique direct efficace. L'enjeu
théorique est : **quelle est la classe** des circuits qu'on peut simuler en **polynomial**
classique ?

## 3. Le résultat de Gottesman-Knill : une description polynomiale

Une famille de portes joue un rôle particulier en informatique quantique : les portes
**Clifford** (Hadamard, phase S, CNOT). Aaronson & Gottesman (2004) ont montré qu'un
circuit construit uniquement avec ces portes se simule classiquement en **polynomial**,
avec un tableau d’état de taille **O(n²)** ; le coût total dépend aussi des portes et des mesures effectuées. L'idée : l'état d'un circuit Clifford
peut être décrit non pas par ses 2^n amplitudes, mais par une **matrice de Pauli** sur
n qubits — un objet de taille O(n²) sur {I, X, Y, Z}^n.

Ce théorème, ce carnet ne le démontre pas : la vraie simulation stabilisatrice — table
de Pauli mise à jour porte par porte — est portée par le carnet jumeau
[06b — Aaronson-Gottesman](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb).
La cellule suivante fait un geste plus modeste, et c'est le geste honnête à cette échelle :
sur le cas concret d'un circuit Clifford à n qubits (Hadamard puis CNOT) dont la
distribution finale de mesure est **uniforme** (à un rephasage près), elle chronomètre un
**échantillonnage direct** — tirer chaque bit selon une distribution connue à l'avance,
en O(n) par échantillon, sans construire le circuit ni maintenir de table de
stabilisateurs. Le banc matérialise la *conséquence* du théorème (quand la structure est
connue, l'échantillon est bon marché), pas sa preuve.

In [3]:
def echantillonnage_direct(n_qubits, n_echantillons=4096):
    """Echantillonnage direct d'une distribution connue analytiquement (uniforme).

    **Ce n'est PAS Gottesman-Knill** : on ne maintient pas de table de
    stabilisateurs, on n'alterne pas de portes Clifford. La fonction tire
    directement n bits i.i.d. uniformes, ce qui equivaut a mesurer
    independamment chaque qubit d'un etat |+>^n (apres un rephasage).

    Gottesman-Knill PROPREMENT DIT (matrice de Pauli O(n^2) sur n qubits,
    O(n^3) pour l'extraction d'une distribution d'echantillons) est implemente
    dans le carnet 06b (Aaronson-Gottesman, simulation par stabilisateurs) ;
    ici on ne mesure que le coût de l'echantillonnage direct d'une distribution
    CONNUE, qui sert de point de comparaison (O(n) par echantillon).
    """
    rng = np.random.default_rng(seed=20261007)
    # Warm-up isole
    _ = rng.integers(0, 2, size=(64, n_qubits), dtype=np.int8)
    t0 = time.perf_counter()
    bits = rng.integers(0, 2, size=(n_echantillons, n_qubits), dtype=np.int8)
    t1 = time.perf_counter()
    return t1 - t0, bits

for n in (4, 8, 16, 20, 24, 28):
    t, _ = echantillonnage_direct(n, n_echantillons=4096)
    print(f"n = {n:>2} qubits : {t*1000:7.2f} ms pour 4096 echantillons (echantillonnage direct : O(n), distribution connue)")

n =  4 qubits :    0.04 ms pour 4096 echantillons (echantillonnage direct : O(n), distribution connue)
n =  8 qubits :    0.14 ms pour 4096 echantillons (echantillonnage direct : O(n), distribution connue)
n = 16 qubits :    0.12 ms pour 4096 echantillons (echantillonnage direct : O(n), distribution connue)
n = 20 qubits :    0.15 ms pour 4096 echantillons (echantillonnage direct : O(n), distribution connue)
n = 24 qubits :    0.29 ms pour 4096 echantillons (echantillonnage direct : O(n), distribution connue)
n = 28 qubits :    0.20 ms pour 4096 echantillons (echantillonnage direct : O(n), distribution connue)


Le coût par bit est constant dans cette implémentation directe, et le coût total est O(n) par échantillon : aucune
construction de 2^n amplitudes, aucune matrice 2^n × 2^n. Cette loi est uniforme et connue analytiquement : le tirage direct est exact dans ce cas particulier. Ce banc ne caractérise pas les circuits corrélés généraux ni la difficulté du boson sampling. Le coût total du tirage direct croît avec le nombre de bits produits, O(n) par échantillon.

## 4. Au-delà de Clifford : la porte T et la question BQP/P

Le jeu de portes Clifford complété par T = diag(1, exp(iπ/4)) est universel, avec une précision d’approximation donnée. Un circuit particulier contenant un T ne devient pas à lui seul universel ni nécessairement difficile à simuler. La question de la simulation polynomiale d'un
tel circuit est précisément la **frontière BQP/P** :

    BQP ⊆ P   ?   (ouvert)

où **BQP** (Bounded-Error Quantum Polynomial time) est la classe des problèmes qu'un
ordinateur quantique résout en temps polynomial avec probabilité d'erreur bornée. Comme P ⊆ BQP, une inclusion BQP ⊆ P donnerait l’égalité pour les problèmes de décision. Si cette inclusion est fausse,
il existe des problèmes que le quantique résout et le classique ne sait pas résoudre en
polynomial.

La simulation vectorielle générale d’un circuit à n qubits et profondeur p coûte **O(2^n · poly(n, p))** — sans avancée algorithmique, cette
barrière reste exponentielle. C'est cette barrière qui motive la promesse algorithmique
du quantique ; c'est aussi elle qui rend la simulation classique **utilisable** sur les
cas Clifford, et rapidement coûteuse pour de grands circuits sans structure exploitable ; de petits circuits restent simulables.
La nature de l’algorithme classique compte : décision déterministe polynomiale pour tout BQP implique P = BQP ; simulation probabiliste fidèle polynomiale implique BPP = BQP. Ces deux conclusions ne sont pas interchangeables, et ce banc ne démontre aucune des deux.


## Exercices

Les trois exercices suivent le canevas : **mesure → compréhension → conjecture**. Le
lecteur complète l'implémentation et consigne observations et chiffrements dans les
cellules marquées `# TODO etudiant`.

### Exercice 1 — Décompte #P de la simulation universelle

On a vu qu'un circuit quantique produit une distribution de mesure sur 2^n chaînes. Pour
un circuit universel arbitraire, **calculer exactement cette distribution** est #P-dur
en général (le carnet [05b](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb)
le démontre pour le boson sampling via la permanente).

**Travail** :
1. Choisir un circuit jouet de 4 qubits utilisant H, T et CNOT ;
2. Construire la matrice unitaire complète (taille 16 × 16) ;
3. Calculer les 16 probabilités en sortie (vecteur d'état × mesure) ;
4. Vérifier que la somme des probabilités vaut 1, et observer la **distribution**
   (est-elle proche de l'uniforme, piquée sur quelques chaînes, autre ?) ;
5. Comparer au temps de calcul de la simulation Gottesman-Knill sur ce même circuit
   (cas dégénéré : pas de porte T → la simulation polynomiale doit suffire).

Le tableau stabilisateur ne traite pas directement T. Cela ne prouve ni qu’un T impose un coût exponentiel, ni que le circuit particulier est universel.

### Exercice 2 — Distinction échantillonner / calculer (Aaronson-Arkhipov)

**Travail** :
1. Lire (ou résumer) le résultat Aaronson-Arkhipov (2011) : la distinction
   **échantillonner vs calculer** pour le boson sampling est le paradigme canonique
   de la frontière calculable / échantillonnable classiquement ; pour un circuit
   quantique UNIVERSEL sur n qubits, **calculer la distribution exacte** est
   généralement #P-dur (cf carnet 05b), tandis qu'**échantillonner** peut rester
   faisable en O(n) par échantillon si la structure du circuit le permet ;
2. Échantillonner depuis la distribution exacte calculée à l'exercice 1 : après le coût
   de **construction** (matrice 16 × 16 et distribution calculées une fois pour toutes),
   mesurer le coût de **tirage** — temps pour 1000 échantillons tirés dans cette
   distribution — et le comparer à l'échantillonnage direct du §3 sur Clifford
   (O(n) par échantillon, aucun coût de construction) ;
3. Conjecturer : pour quel type de problème (échantillonner vs calculer la distribution)
   la simulation classique est-elle acceptable en pratique ?

### Exercice 3 — La frontière BQP/P

**Travail** :
1. Énoncer (en deux phrases) la définition de BQP ;
2. Rappeler (en une phrase) ce que la barrière O(2^n) classique nous dit sur
   la séparation BQP vs P / PH (rappel : BQP ⊆ PP ⊆ P^#P est un théorème ;
   **l'ouverte** est BQP vs P / PH, pas l'inclusion BQP ⊂ P^#P) ;
3. Conjecturer (en trois phrases maximum) : si demain un algorithme classique en
   **O(poly(n))** pour les circuits universels était trouvé, que deviendrait la valeur
   algorithmique du calcul quantique ?
Distinguez dans votre raisonnement décision déterministe et simulation probabiliste : la première hypothèse concerne P, la seconde BPP. Définir BQP inclut un temps polynomial et une erreur bornée sur les deux types d’instances, amplifiable par répétition.


In [4]:
# === Cellule de verification pour les exercices ===
#
# Les trois cellules ci-dessous sont les points d'entree pour tester vos implementations.
# Elles DOIVENT s'executer sans erreur meme si les implementations sont incompletes :
# chaque bloc TODO rend None : le harnais rapporte alors l'absence de solution
# au lieu de lever une exception volontaire (regle C.1).

import sys

def _safe_run(label, fn):
    try:
        resultat = fn()
        if resultat is None:
            print(f"[todo]   {label} : a completer")
        else:
            print(f"[ok]     {label}")
    except Exception as e:
        print(f"[err]    {label} : {type(e).__name__}: {e}")


# --- Exercice 1 : distribution d'un circuit universel a 4 qubits ---
def exercice_1():
    # TODO etudiant : construire un circuit H, T, CNOT sur 4 qubits, calculer la matrice
    # unitaire 16x16 et la distribution de probabilite de mesure sur la base
    # computationnelle.
    return None  # TODO etudiant

_safe_run("Exercice 1 - distribution circuit universel 4 qubits", exercice_1)


# --- Exercice 2 : echantillonner vs calculer (Aaronson-Arkhipov) ---
def exercice_2():
    # TODO etudiant : distinguer le cout de construction (matrice et distribution exactes
    # de l'exercice 1, calculees une fois) du cout de tirage (1000 echantillons dans cette
    # distribution), face a l'echantillonnage direct de la section 3 (O(n) par echantillon,
    # sans construction).
    return None  # TODO etudiant

_safe_run("Exercice 2 - echantillonner vs calculer (Aaronson-Arkhipov)", exercice_2)


# --- Exercice 3 : enonce BQP, conjecturer la frontiere BQP/P ---
def exercice_3():
    # TODO etudiant : formuler (en texte) la definition de BQP, rappeler la barriere
    # O(2^n) et conjecturer sur l'avenir algorithmique du calcul quantique.
    return None  # TODO etudiant

_safe_run("Exercice 3 - BQP / P", exercice_3)

[todo]   Exercice 1 - distribution circuit universel 4 qubits : a completer
[todo]   Exercice 2 - echantillonner vs calculer (Aaronson-Arkhipov) : a completer
[todo]   Exercice 3 - BQP / P : a completer


## À retenir

Ce carnet a posé la frontière de la simulation classique d'un circuit quantique par le
biais de deux résultats canoniques :

- le coût exponentiel O(2^n) de la représentation vectorielle explicite, qui sépare la
  zone où la simulation classique naïve reste praticable (n petit) de celle où elle
  devient intraitable (n ≥ 30) ;
- l'**algorithme de Gottesman-Knill** (1998), qui montre que la sous-famille **Clifford**
  (H, S, CNOT) admet une description d’état O(n²) et une simulation polynomiale, donc compatible avec une
  simulation classique efficace des portes et mesures stabilisatrices, sans matérialiser toutes les probabilités.
Le pont avec la **déquantification Aaronson-Gottesman** et la **stabilisation** est
approfondi dans le carnet [06b](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb).

**Sources** :
- Aaronson & Arkhipov, *The Computational Complexity of Linear Optics*, Theory of Computing 9 (2013), pp. 143-252 (boson sampling : distinction echantillonner / calculer) ;
- Aaronson & Gottesman, *Improved Simulation of Stabilizer Circuits*, Phys. Rev. A 70 (2004) ;
- Nielsen & Chuang, *Quantum Computation and Quantum Information*, 10th anniv. ed. (2010),
  ch. 10 sur la stabilité et la simulation ;
- Aaronson, *BQP and the Polynomial Hierarchy* (2009), pour la conjecture BQP ⊄ PH.

## Vérifiez votre compréhension

1. Vers $n \approx 28$–$30$, le vecteur d'état déborde la mémoire d'un poste standard ($2^{30} \times 16$ octets $\approx 17{,}2$ Go). Pourquoi ce coût dépend-t-il **seulement** de $n$ — et pas de la profondeur du circuit ni du nombre de portes ?
2. L'échantillonnage direct du §3 est quasi-constant par qubit (4 à 28 qubits, même ordre de temps) — et le carnet insiste : ce n'est **pas** Gottesman-Knill. Que mesure-t-il exactement, et où vit le vrai Gottesman-Knill ?
3. Le jeu Clifford+T est universel, mais un seul T ne rend pas chaque circuit difficile ; $\mathrm{BQP} \subseteq P$ reste ouvert. Qu'est-ce que le théorème de Gottesman-Knill établit précisément — et que ne dit-il **pas** sur le pouvoir du quantique ?

<details><summary>Réponses</summary>

1. Parce que l'état complet est stocké comme un vecteur de $2^n$ amplitudes, quelle que soit la séquence de portes qui l'a produit : chaque porte n'est qu'une matrice appliquée à ce vecteur. La mémoire paie la **description** de l'état, pas le circuit qui l'engendre — le tableau stabilisateur du 06b occupe lui aussi une mémoire dépendant de n, O(n²), et ses opérations dépendent des portes et mesures.
2. Il mesure le coût d'échantillonner une distribution **connue analytiquement** (uniforme : $n$ bits i.i.d., $O(n)$ par échantillon) — un point de comparaison de plancher, aucune table de stabilisateurs. La simulation par stabilisateurs — tableau de taille $O(n^2)$ et mise à jour des portes/mesures — est implémenté dans le carnet 06b.
3. Il établit que la sous-famille **Clifford** (H, S, CNOT) se simule en polynomial classique : cette terre-là ne demande pas de moteur quantique. Il ne dit rien des circuits universels : ajouter $T$ sort du seul jeu Clifford, sans prouver une difficulté exponentielle ; la méthode vectorielle générale coûte $O(2^n \cdot \mathrm{poly}(n, p))$. La question de fond — existe-t-il du quantique non simulable — reste exactement là.

</details>

Distinguez dans votre raisonnement décision déterministe et simulation probabiliste : la première hypothèse concerne P, la seconde BPP. Définir BQP inclut un temps polynomial et une erreur bornée sur les deux types d’instances, amplifiable par répétition.


## Pour aller plus loin

- **Carnet suivant** : [Complexity-06b — Aaronson-Gottesman : déquantification](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb) — l'arbitre par stabilisateurs qui exécute le cas Clifford en $O(n^2)$ et refuse poliment la première porte $T$.
- **Dans la même série** : [Complexity-06c — Classical at heart](Complexity-06c-ClassicalAtHeart-Python.ipynb) — la question miroir : implémenter une opération quantique à partir d'un oracle classique ; [Complexity-05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) — quand le mur du comptage devient un candidat d'avantage.
- **Carnet voisin d'une autre série** : [Lean-13b — La borne de Tsirelson](../SymbolicAI/Lean/Lean-13b-CHSH-Tsirelson-Native.ipynb) — la même frontière classique/quantique, cette fois digérée en Lean : le jeu de CHSH borne le score du classique à 2 et celui de l'intriqué à $2\sqrt{2}$.
- **Lectures** : les sources (Aaronson-Arkhipov 2013, Aaronson-Gottesman 2004, Nielsen-Chuang ch. 10, Aaronson 2009) figurent en fin de section « À retenir » ; la source Aaronson-Arkhipov (arXiv 1011.3245) est archivée au gisement partagé (dossier `Bibliographie IA/Consciousness/`), les trois autres références n'y ont pas de copie à ce jour.
